In [ ]:
# Cell 1: Install Libraries
print("Installing required libraries...")
!pip install tf_keras transformers -q
print("Done! Please RESTART THE RUNTIME before running the next cell.")

Installing required libraries...
Done! Please RESTART THE RUNTIME before running the next cell.


In [ ]:
# Cell 2: Set Keras 2, Import Libraries & Check GPU
# This line MUST run before importing tensorflow
import os
os.environ['TF_USE_LEGACY_KERAS'] = 'True'

import tensorflow as tf
from google.colab import drive
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import numpy as np

In [ ]:
# Cell 3: Connect to Google Drive
print("Connecting to Google Drive...")
drive.mount('/content/drive')

Connecting to Google Drive...
Mounted at /content/drive


In [ ]:
# Cell 4: Load Model and Tokenizer
# This path MUST match where you saved your model in the last notebook
model_path = "/content/drive/MyDrive/my_roberta_model" # <-- UPDATED

print(f"Loading tokenizer from: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(model_path)

print(f"Loading model from: {model_path}")
model = TFAutoModelForSequenceClassification.from_pretrained(model_path)

print("\nModel and tokenizer loaded successfully!")

Loading tokenizer from: /content/drive/MyDrive/my_roberta_model
Loading model from: /content/drive/MyDrive/my_roberta_model


Transformers is only compatible with Keras 2, but you have explicitly set `TF_USE_LEGACY_KERAS` to `0`. This may result in unexpected behaviour or errors if Keras 3 objects are passed to Transformers models.
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at /content/drive/MyDrive/my_roberta_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.



Model and tokenizer loaded successfully!


In [ ]:
# Cell 5: Create the Prediction Function
def predict_sentiment(review_text):
  # 1. Tokenize the text
  inputs = tokenizer(review_text, return_tensors="tf", truncation=True, padding=True)

  # 2. Make a prediction
  # We use dict(inputs) to feed the model the inputs as a dictionary
  raw_outputs = model.predict(dict(inputs))

  # 3. Get the logits (raw scores)
  logits = raw_outputs.logits[0]

  # 4. Convert logits to probabilities using softmax
  probabilities = tf.nn.softmax(logits).numpy()

  # 5. Get the predicted class (0 or 1)
  prediction = np.argmax(probabilities)

  if prediction == 1:
    sentiment = "Positive"
    confidence = probabilities[1]
  else:
    sentiment = "Negative"
    confidence = probabilities[0]

  return sentiment, confidence * 100

print("Prediction function is ready.")

Prediction function is ready.


In [ ]:
# Cell 6: Test With Your Own Inputs!
my_review = "I wasn't sure what to expect, but the movie was incredible. The story was deep and the actors were perfect. I loved it."
sentiment, confidence = predict_sentiment(my_review)
print(f"Review: '{my_review}'")
print(f"Predicted Sentiment: {sentiment}")
print(f"Confidence: {confidence:.2f}%")

print("\n" + "-"*30 + "\n")

my_other_review = "A complete waste of time. The plot made no sense and I was bored."
sentiment, confidence = predict_sentiment(my_other_review)
print(f"Review: '{my_other_review}'")
print(f"Predicted Sentiment: {sentiment}")
print(f"Confidence: {confidence:.2f}%")

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


1/1 [==============================] - 9s 9s/step
Review: 'I wasn't sure what to expect, but the movie was incredible. The story was deep and the actors were perfect. I loved it.'
Predicted Sentiment: Positive
Confidence: 99.52%

------------------------------

1/1 [==============================] - 3s 3s/step
Review: 'A complete waste of time. The plot made no sense and I was bored.'
Predicted Sentiment: Negative
Confidence: 99.85%


In [ ]:
# Cell 7: Interactive CLI for Predictions
print("--- Starting Interactive Sentiment Analyzer ---")
print("Type your movie review and press Enter.")
print("Type 'quit', 'exit', or 'q' to stop.")
print("-" * 45)

while True:
  # Get input from the user
  review_text = input("\nEnter your review: ")

  # Check for an exit command
  if review_text.lower() in ['quit', 'exit', 'q']:
    print("\nExiting analyzer. Goodbye!")
    break

  # Make sure the user actually typed something
  if not review_text.strip():
    print("Please enter some text to analyze.")
    continue

  # Call the prediction function
  try:
    sentiment, confidence = predict_sentiment(review_text)

    # Print the result
    print(f"\n  -> Predicted Sentiment: {sentiment}")
    print(f"  -> Confidence: {confidence:.2f}%")
    print("-" * 45)

  except Exception as e:
    print(f"\nAn error occurred during prediction: {e}")
    print("Please try again.")
    print("-" * 45)

--- Starting Interactive Sentiment Analyzer ---
Type your movie review and press Enter.
Type 'quit', 'exit', or 'q' to stop.
---------------------------------------------

Enter your review: The movie was good.
1/1 [==============================] - 0s 102ms/step

  -> Predicted Sentiment: Positive
  -> Confidence: 55.32%
---------------------------------------------

Enter your review: The movie was too good, I'll recommend it to others too.
1/1 [==============================] - 0s 132ms/step

  -> Predicted Sentiment: Positive
  -> Confidence: 91.73%
---------------------------------------------

Enter your review: q

Exiting analyzer. Goodbye!
